## **FAS Inference: Load the best model (Model 1) from its saved checkpoints.**

In [1]:
import os

# Must be set before importing ONNX Runtime, PyTorch,
# NumPy or other libraries that create worker threads.
os.environ["ORT_DISABLE_THREAD_AFFINITY"] = "1"
os.environ["ORT_LOG_SEVERITY_LEVEL"] = "3"
os.environ["OMP_NUM_THREADS"] = "4"
os.environ["MKL_NUM_THREADS"] = "4"

import warnings
warnings.filterwarnings("ignore")

import logging
logging.getLogger().setLevel(logging.ERROR)

In [2]:
%load_ext autoreload
%autoreload 3

In [3]:
# !pip install mediapipe

In [4]:
# ============================================================
# STEP 0 — INSTALL CHALLENGE–RESPONSE APP DEPENDENCIES
# ============================================================
#
# %pip installs packages into the Python environment used by
# the current Jupyter kernel.
#
# PyTorch and torchvision are intentionally excluded because
# the existing FAS model environment should remain unchanged.
# ============================================================

# %pip install --upgrade \
#     gradio \
#     mediapipe \
#     opencv-python \
#     scipy \
#     pydantic

In [5]:
!pip show mediapipe

Name: mediapipe
Version: 0.10.35
Summary: MediaPipe is the simplest way for researchers and developers to build world-class ML solutions and applications for mobile, edge, cloud and the web.
Home-page: https://github.com/google/mediapipe
Author: The MediaPipe Authors
Author-email: mediapipe@google.com
License: Apache 2.0
Location: /storage/research/csdata_L/xp814142/uvenvs/my_environment/lib64/python3.11/site-packages
Requires: absl-py, certifi, flatbuffers, matplotlib, numpy, opencv-contrib-python, sounddevice
Required-by: 


In [5]:
# ============================================================================
#  STEP 1  - IMPORTS - covers every component of the challenge-response pipeline
# ============================================================================
from __future__ import annotations
from lib_imports import *                        # existing training-pipeline imports (torch, numpy, cv2, Path, etc.)
print("All required libraries imported successfully.\n")
#------------------------

#-----------------------
print("=" * 50)
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA version:", torch.version.cuda)
    print("CUDA device count:", torch.cuda.device_count())
    print("Current GPU:", torch.cuda.get_device_name(0))
    print("cuDNN version:", torch.backends.cudnn.version())
print()
print("ONNXRuntime version:", ort.__version__)
print("ONNX providers:", ort.get_available_providers())
print("=" * 50)

# Automatic device detection
RUNTIME_MODE = "auto"        # Options: "auto", "cpu", "gpu", "mps"

# auto — pick the best available device
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")                 # NVIDIA GPU
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")                  # Apple Silicon GPU
else:
    DEVICE = torch.device("cpu")                  # Fallback to CPU

# Print device name
print("Using device:", DEVICE)
if DEVICE.type == "cuda":
    print("Device name:", torch.cuda.get_device_name(0))
elif DEVICE.type == "mps":
    print("Device name: Apple Silicon GPU (MPS)")
else:
    print("Device name: CPU")

All required libraries imported successfully.

PyTorch version: 2.5.1+cu121
CUDA available: True
CUDA version: 12.1
CUDA device count: 1
Current GPU: NVIDIA H100 NVL
cuDNN version: 90100

ONNXRuntime version: 1.26.0
ONNX providers: ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']
Using device: cuda
Device name: NVIDIA H100 NVL


<div style="background-color: #FFF9C4; padding: 12px; border-radius: 6px;">
<hr style="border: none; height: 3px; background-color: #FFD600;">
</div>

In [8]:
#========================================================
# Reproducibility Setup
#========================================================

# Function to control randomness across libraries
def set_seed(seed):                                          
    random.seed(seed)                               # Fix Python random numbers
    np.random.seed(seed)                            # Fix NumPy random numbers
    torch.manual_seed(seed)                         # Fix PyTorch CPU random numbers
    torch.cuda.manual_seed_all(seed)                # Fix PyTorch GPU random numbers
    torch.backends.cudnn.deterministic = True       # Ensure deterministic GPU operations (Forces same operations every time)
    torch.backends.cudnn.benchmark = False          # Disable auto-optimisation for consistent results (Stops CUDA from picking fastest algorithms, which can be random)
    print(f"Seeding applied: SEED={SEED}")

SEED = 42  # Set a fixed seed value for reproducibility
set_seed(SEED)

Seeding applied: SEED=42


<div style="background-color: #FFF9C4; padding: 12px; border-radius: 6px;">
<hr style="border: none; height: 3px; background-color: #FFD600;">
</div>

In [9]:
# ============================================================================
#  STEP 2 — Define paths
# ============================================================================
from lib_imports import *

# Define current notebook path
PROJECT_ROOT = Path.cwd().resolve()

# Existing trained FAS checkpoint
# CKPT_PATH = Path('./output_combinedData/model_1/best_model_combined.pth')
CHECKPOINT_PATH = PROJECT_ROOT / 'output_combinedData' / 'model_1' / 'best_model_combined.pth'

# Challenge Response Output Directories
OUTPUT_DIR = PROJECT_ROOT / "output_challengeResponse"
FACE_LANDMARKER_DIR = PROJECT_ROOT / "output_challengeResponse" / "media_face"     # Mediapipe face folder
TEMP_DIR = OUTPUT_DIR / "temp"
LOG_DIR = OUTPUT_DIR / "logs"
EVIDENCE_DIR = OUTPUT_DIR / "manual_review"

# Create output folders if they do not already exist.
for directory in (
    FACE_LANDMARKER_DIR,
    TEMP_DIR,
    LOG_DIR,
    EVIDENCE_DIR,
):
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# MediaPipe file path - this file will be added in a later step.
FACE_LANDMARKER_FILE_PATH = (
    PROJECT_ROOT / "output_challengeResponse" / "media_face" / "face_landmarker.task"
)

# Display configured paths
print("=" * 70)
print("PROJECT PATH CONFIGURATION")
print("=" * 70)
print(f"Project root       : {PROJECT_ROOT}")
print(f"Checkpoint         : {CHECKPOINT_PATH}")
print(f"Output directory   : {OUTPUT_DIR}")
print(f"MediaPipe model    : {FACE_LANDMARKER_DIR}")
print(f"Temporary files    : {TEMP_DIR}")
print(f"Logs               : {LOG_DIR}")
print(f"Manual review      : {EVIDENCE_DIR}")
print("=" * 70)

# Check whether the existing checkpoint path is valid
if CHECKPOINT_PATH.is_file():
    print("Checkpoint found successfully.")
else:
    print(
        "\nCheckpoint not found at the configured path.\n"
        "Update CHECKPOINT_PATH to the correct checkpoint file."
    )

PROJECT PATH CONFIGURATION
Project root       : /storage/research/csdata_L/xp814142/MSc_Project
Checkpoint         : /storage/research/csdata_L/xp814142/MSc_Project/output_combinedData/model_1/best_model_combined.pth
Output directory   : /storage/research/csdata_L/xp814142/MSc_Project/output_challengeResponse
MediaPipe model    : /storage/research/csdata_L/xp814142/MSc_Project/output_challengeResponse/media_face
Temporary files    : /storage/research/csdata_L/xp814142/MSc_Project/output_challengeResponse/temp
Logs               : /storage/research/csdata_L/xp814142/MSc_Project/output_challengeResponse/logs
Manual review      : /storage/research/csdata_L/xp814142/MSc_Project/output_challengeResponse/manual_review
Checkpoint found successfully.


<div style="background-color: #FFF9C4; padding: 12px; border-radius: 6px;">
<hr style="border: none; height: 3px; background-color: #FFD600;">
</div>

In [10]:
# ============================================================
# STEP 3 — INSPECT THE SAVED FAS CHECKPOINT
# ============================================================
#
# Purpose:
#   - Confirm that the checkpoint can be opened
#   - Identify whether it contains:
#       1. A plain model state_dict
#       2. A training checkpoint with model_state_dict
#       3. Another checkpoint structure
#
# The checkpoint is loaded onto CPU only for inspection.
# ============================================================


def is_model_state_dict(value: Any) -> bool:
    """
    Check whether an object looks like a PyTorch model state_dict.

    A state_dict normally contains:
        - String parameter names as keys
        - PyTorch tensors as values
    """

    return (
        isinstance(value, dict)
        and len(value) > 0
        and all(isinstance(key, str) for key in value.keys())
        and all(torch.is_tensor(item) for item in value.values())
    )


# ------------------------------------------------------------
# Verify that the configured checkpoint exists
# ------------------------------------------------------------

if not CHECKPOINT_PATH.is_file():
    raise FileNotFoundError(
        f"Checkpoint not found:\n{CHECKPOINT_PATH.resolve()}"
    )


# ------------------------------------------------------------
# Load checkpoint safely onto CPU
# ------------------------------------------------------------
# weights_only=False is required because this checkpoint
# contains additional Python/NumPy objects besides tensors.
checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False,
)


# ------------------------------------------------------------
# Display basic checkpoint information
# ------------------------------------------------------------

print("=" * 70)
print("CHECKPOINT INSPECTION")
print("=" * 70)

print(f"Checkpoint path : {CHECKPOINT_PATH.resolve()}")
print(f"Checkpoint type : {type(checkpoint).__name__}")


# ------------------------------------------------------------
# Detect checkpoint structure
# ------------------------------------------------------------

if is_model_state_dict(checkpoint):

    CHECKPOINT_FORMAT = "plain_state_dict"
    MODEL_STATE_DICT = checkpoint

    print("\nDetected format: Plain model state_dict")
    print(f"Number of tensors: {len(MODEL_STATE_DICT)}")


elif isinstance(checkpoint, dict):

    print("\nTop-level checkpoint keys:")

    for key in checkpoint.keys():
        value = checkpoint[key]

        if isinstance(value, dict):
            description = f"dict with {len(value)} entries"

        elif torch.is_tensor(value):
            description = (
                f"Tensor, shape={tuple(value.shape)}, dtype={value.dtype}"
            )

        else:
            description = type(value).__name__

        print(f"  - {key}: {description}")


    # --------------------------------------------------------
    # Search common locations used to store model weights
    # --------------------------------------------------------

    candidate_keys = (
        "model_state_dict",
        "state_dict",
        "model",
        "model_weights",
    )

    MODEL_STATE_DICT = None
    CHECKPOINT_FORMAT = "unknown_dictionary"

    for candidate_key in candidate_keys:

        candidate_value = checkpoint.get(candidate_key)

        if is_model_state_dict(candidate_value):

            MODEL_STATE_DICT = candidate_value
            CHECKPOINT_FORMAT = candidate_key

            print(
                f"\nModel weights found under key: "
                f"'{candidate_key}'"
            )

            print(
                f"Number of model tensors: "
                f"{len(MODEL_STATE_DICT)}"
            )

            break


    if MODEL_STATE_DICT is None:
        print(
            "\nNo model state_dict was automatically found "
            "under the common checkpoint keys."
        )


else:

    CHECKPOINT_FORMAT = "unsupported"
    MODEL_STATE_DICT = None

    print(
        "\nThe checkpoint is not a dictionary or plain state_dict."
    )


# ------------------------------------------------------------
# Display a small sample of parameter names
# ------------------------------------------------------------

if MODEL_STATE_DICT is not None:

    print("\nFirst 10 parameter names:")

    for parameter_name in list(MODEL_STATE_DICT.keys())[:10]:
        tensor = MODEL_STATE_DICT[parameter_name]

        print(
            f"  - {parameter_name:<55} "
            f"shape={tuple(tensor.shape)}"
        )


print("\n" + "=" * 70)
print(f"Detected checkpoint format: {CHECKPOINT_FORMAT}")
print("=" * 70)

CHECKPOINT INSPECTION
Checkpoint path : /share/nas/cs-nas/xp814142/output_combinedData/model_1/best_model_combined.pth
Checkpoint type : dict

Top-level checkpoint keys:
  - epoch: int
  - state_dict: dict with 836 entries
  - optimizer_state_dict: dict with 2 entries
  - scaler_state_dict: dict with 5 entries
  - val_acer: float64
  - val_metrics: dict with 5 entries
  - history: dict with 14 entries

Model weights found under key: 'state_dict'
Number of model tensors: 836

First 10 parameter names:
  - cnn_rgb.backbone.stem.0.weight                          shape=(96, 3, 4, 4)
  - cnn_rgb.backbone.stem.0.bias                            shape=(96,)
  - cnn_rgb.backbone.stem.1.weight                          shape=(96,)
  - cnn_rgb.backbone.stem.1.bias                            shape=(96,)
  - cnn_rgb.backbone.stages.0.blocks.0.gamma                shape=(96,)
  - cnn_rgb.backbone.stages.0.blocks.0.conv_dw.weight       shape=(96, 1, 7, 7)
  - cnn_rgb.backbone.stages.0.blocks.0.conv_dw

<div style="background-color: #FFF9C4; padding: 12px; border-radius: 6px;">
<hr style="border: none; height: 3px; background-color: #FFD600;">
</div>

In [11]:
# ============================================================
# STEP 4 — PREPARE A CLEAN MODEL STATE_DICT
# ============================================================
#
# This cell:
#   1. Confirms that model weights were found
#   2. Creates a separate copy of the weights
#   3. Removes a possible "module." prefix added by
#      DataParallel or DistributedDataParallel
#   4. Performs basic integrity checks
#
# It does not load the weights into the model yet.
# ============================================================

from copy import deepcopy

from torch.nn.modules.utils import (
    consume_prefix_in_state_dict_if_present,
)


# ------------------------------------------------------------
# Confirm that Step 3 found the model weights
# ------------------------------------------------------------
if MODEL_STATE_DICT is None:
    raise ValueError(
        "MODEL_STATE_DICT is empty.\n"
        "Review the checkpoint keys printed in Step 4."
    )


# ------------------------------------------------------------
# Copy the state_dict:
# A copy is used so that the original checkpoint object remains
# unchanged in memory.
# ------------------------------------------------------------
CLEAN_MODEL_STATE_DICT = deepcopy(MODEL_STATE_DICT)


# ------------------------------------------------------------
# Check whether the checkpoint was saved from DataParallel/DDP
# Such checkpoints commonly store parameter names as:
#   module.rgb_branch.conv1.weight
# A normal single-model instance usually expects:
#   rgb_branch.conv1.weight
# ------------------------------------------------------------
parameter_names_before = list(CLEAN_MODEL_STATE_DICT.keys())

has_module_prefix = any(
    name.startswith("module.")
    for name in parameter_names_before
)


if has_module_prefix:
    consume_prefix_in_state_dict_if_present(
        CLEAN_MODEL_STATE_DICT,
        prefix="module.",
    )

    print("Removed the 'module.' prefix from parameter names.")

else:
    print("No 'module.' prefix was found.")


# ------------------------------------------------------------
# Basic state_dict integrity checks
# ------------------------------------------------------------

if len(CLEAN_MODEL_STATE_DICT) == 0:
    raise ValueError("The cleaned state_dict is empty.")

non_tensor_entries = [
    name
    for name, value in CLEAN_MODEL_STATE_DICT.items()
    if not torch.is_tensor(value)
]

if non_tensor_entries:
    raise TypeError(
        "The model state_dict contains non-tensor entries:\n"
        + "\n".join(non_tensor_entries[:10])
    )

# ------------------------------------------------------------
# Display the cleaned state_dict summary
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("CLEAN MODEL STATE_DICT")
print("=" * 70)

print(
    f"Checkpoint format : {CHECKPOINT_FORMAT}"
)

print(
    f"Number of tensors : {len(CLEAN_MODEL_STATE_DICT)}"
)

print("\nFirst 10 parameter names:")

for parameter_name in list(
    CLEAN_MODEL_STATE_DICT.keys()
)[:10]:

    tensor = CLEAN_MODEL_STATE_DICT[parameter_name]

    print(
        f"  {parameter_name:<60} "
        f"shape={tuple(tensor.shape)}, "
        f"dtype={tensor.dtype}"
    )

print("=" * 70)

No 'module.' prefix was found.

CLEAN MODEL STATE_DICT
Checkpoint format : state_dict
Number of tensors : 836

First 10 parameter names:
  cnn_rgb.backbone.stem.0.weight                               shape=(96, 3, 4, 4), dtype=torch.float32
  cnn_rgb.backbone.stem.0.bias                                 shape=(96,), dtype=torch.float32
  cnn_rgb.backbone.stem.1.weight                               shape=(96,), dtype=torch.float32
  cnn_rgb.backbone.stem.1.bias                                 shape=(96,), dtype=torch.float32
  cnn_rgb.backbone.stages.0.blocks.0.gamma                     shape=(96,), dtype=torch.float32
  cnn_rgb.backbone.stages.0.blocks.0.conv_dw.weight            shape=(96, 1, 7, 7), dtype=torch.float32
  cnn_rgb.backbone.stages.0.blocks.0.conv_dw.bias              shape=(96,), dtype=torch.float32
  cnn_rgb.backbone.stages.0.blocks.0.norm.weight               shape=(96,), dtype=torch.float32
  cnn_rgb.backbone.stages.0.blocks.0.norm.bias                 shape=(96,), dty

<div style="background-color: #FFF9C4; padding: 12px; border-radius: 6px;">
<hr style="border: none; height: 3px; background-color: #FFD600;">
</div>

In [13]:
# ============================================================
# STEP 5 — CREATE THE MODEL AND LOAD TRAINED WEIGHTS
# ============================================================

from Functions.full_model_pipeline_13 import *

# Build a fresh model object
fas_model = HybridPADModel().to(DEVICE)

# ------------------------------------------------------------
# Safety check
# ------------------------------------------------------------

if "fas_model" not in globals():
    raise NameError(
        "fas_model has not been created.\n"
        "Import your model class and instantiate it above using "
        "the exact training configuration."
    )


# ------------------------------------------------------------
# Load the trained parameters
#
# strict=True requires the checkpoint parameter names to match
# the instantiated model architecture exactly.
# ------------------------------------------------------------

load_result = fas_model.load_state_dict(
    CLEAN_MODEL_STATE_DICT,
    strict=True,
)


# ------------------------------------------------------------
# Move the model to the selected device and set inference mode
# ------------------------------------------------------------

fas_model = fas_model.to(DEVICE)
fas_model.eval()


# ------------------------------------------------------------
# Confirm successful loading
# ------------------------------------------------------------

print("=" * 70)
print("FAS MODEL LOADED SUCCESSFULLY")
print("=" * 70)

print(f"Model class       : {fas_model.__class__.__name__}")
print(f"Model device      : {next(fas_model.parameters()).device}")
print(f"Training mode     : {fas_model.training}")
print(f"Missing keys      : {load_result.missing_keys}")
print(f"Unexpected keys   : {load_result.unexpected_keys}")

total_parameters = sum(
    parameter.numel()
    for parameter in fas_model.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in fas_model.parameters()
    if parameter.requires_grad
)

print(f"Total parameters  : {total_parameters:,}")
print(f"Trainable params  : {trainable_parameters:,}")

print("=" * 70)

FAS MODEL LOADED SUCCESSFULLY
Model class       : HybridPADModel
Model device      : cuda:0
Training mode     : False
Missing keys      : []
Unexpected keys   : []
Total parameters  : 57,967,392
Trainable params  : 45,617,856


<div style="background-color: #FFF9C4; padding: 12px; border-radius: 6px;">
<hr style="border: none; height: 3px; background-color: #FFD600;">
</div>

In [14]:
# ============================================================
# STEP 6 — INSPECT THE MODEL FORWARD METHOD
# ============================================================
#
# Purpose:
#   - Identify how many inputs the model expects
#   - Display the input parameter names
#   - Check the model's forward-method signature
#
# No inference is performed in this step.
# ============================================================

import inspect

# ------------------------------------------------------------
# Confirm that the model exists and is in inference mode
# ------------------------------------------------------------

if "fas_model" not in globals():
    raise NameError(
        "fas_model was not found. Complete Step 6 first."
    )

if fas_model.training:
    raise RuntimeError(
        "fas_model is still in training mode. "
        "Run fas_model.eval() before continuing."
    )


# ------------------------------------------------------------
# Inspect the forward method
# ------------------------------------------------------------

forward_signature = inspect.signature(
    fas_model.forward
)

forward_parameters = list(
    forward_signature.parameters.values()
)


print("=" * 70)
print("MODEL INPUT CONTRACT")
print("=" * 70)

print(f"Model class  : {fas_model.__class__.__name__}")
print(f"Model module : {fas_model.__class__.__module__}")
print(f"Forward call : forward{forward_signature}")

print("\nExpected forward inputs:")

for index, parameter in enumerate(
    forward_parameters,
    start=1,
):
    print(
        f"  {index}. "
        f"name={parameter.name!r}, "
        f"default={parameter.default}, "
        f"annotation={parameter.annotation}"
    )


# ------------------------------------------------------------
# Count required inputs
# ------------------------------------------------------------

required_parameters = [
    parameter
    for parameter in forward_parameters
    if (
        parameter.default is inspect.Parameter.empty
        and parameter.kind
        not in {
            inspect.Parameter.VAR_POSITIONAL,
            inspect.Parameter.VAR_KEYWORD,
        }
    )
]


print("\n" + "-" * 70)
print(
    f"Number of required forward inputs: "
    f"{len(required_parameters)}"
)

print(
    "Required input names: "
    f"{[parameter.name for parameter in required_parameters]}"
)

print("=" * 70)

MODEL INPUT CONTRACT
Model class  : HybridPADModel
Model module : Functions.full_model_pipeline_13
Forward call : forward(rgb, freq=None, flow=None)

Expected forward inputs:
  1. name='rgb', default=<class 'inspect._empty'>, annotation=<class 'inspect._empty'>
  2. name='freq', default=None, annotation=<class 'inspect._empty'>
  3. name='flow', default=None, annotation=<class 'inspect._empty'>

----------------------------------------------------------------------
Number of required forward inputs: 1
Required input names: ['rgb']


<div style="background-color: #FFF9C4; padding: 12px; border-radius: 6px;">
<hr style="border: none; height: 3px; background-color: #FFD600;">
</div>

In [15]:
# ============================================================
# Step 7 - INSPECT SAVED TEST METRICS JSON
# ============================================================

# Replace this with the actual location of your JSON file.
TEST_METRICS_PATH = PROJECT_ROOT / "output_combinedData" / "model_1" / "held_out_test_metrics.json"

# Validate the file
if not TEST_METRICS_PATH.is_file():
    raise FileNotFoundError(
        f"Test metrics file was not found:\n"
        f"{TEST_METRICS_PATH}"
    )

# Load the JSON file
with TEST_METRICS_PATH.open(
    mode="r",
    encoding="utf-8",
) as file:
    test_metrics = json.load(file)

print("=" * 80)
print("TEST METRICS JSON")
print("=" * 80)
print(f"File path : {TEST_METRICS_PATH}")
print(f"Data type : {type(test_metrics).__name__}")

# Inspect top-level contents
if isinstance(test_metrics, dict):
    print(f"Number of top-level keys: {len(test_metrics)}")
    print("\nTop-level keys and value types:\n")
    for key, value in test_metrics.items():
        print(
            f"{key!r:<35} "
            f"type={type(value).__name__}"
        )
elif isinstance(test_metrics, list):
    print(f"Number of entries: {len(test_metrics)}")
else:
    print(f"Stored value: {test_metrics}")

# Print the complete JSON in readable form
print("\n" + "-" * 80)
print("COMPLETE JSON CONTENT")
print("-" * 80)
print(
    json.dumps(
        test_metrics,
        indent=4,
        ensure_ascii=False,
    )
)
print("=" * 80)

# ============================================================
# EXTRACT THE VALIDATION-SELECTED DECISION THRESHOLD
# ============================================================

if not isinstance(test_metrics, dict):
    raise TypeError("Expected test_metrics.json to contain a dictionary.")

if "best_threshold" not in test_metrics:
    raise KeyError("'best_threshold' was not found in the test-metrics JSON.")

VALIDATION_THRESHOLD = float(test_metrics["best_threshold"])

print("DECISION THRESHOLD")
print("=" * 70)
print(f"Validation threshold: {VALIDATION_THRESHOLD}")
print("=" * 70)

TEST METRICS JSON
File path : /storage/research/csdata_L/xp814142/MSc_Project/output_combinedData/model_1/held_out_test_metrics.json
Data type : dict
Number of top-level keys: 7

Top-level keys and value types:

'evaluation'                        type=str
'best_threshold'                    type=float
'test_metrics'                      type=dict
'checkpoint_epoch'                  type=int
'n_test_videos'                     type=int
'n_test_real'                       type=int
'n_test_fake'                       type=int

--------------------------------------------------------------------------------
COMPLETE JSON CONTENT
--------------------------------------------------------------------------------
{
    "evaluation": "held_out_test (phone 6 + replay)",
    "best_threshold": 0.16464646464646465,
    "test_metrics": {
        "APCER": 0.0216138328527145,
        "BPCER": 0.0933333333271111,
        "ACER": 0.0574735830899128,
        "Acc": 0.9656398104265402,
        "AUC": 0.98

<div style="background-color: #FFF9C4; padding: 12px; border-radius: 6px;">
<hr style="border: none; height: 3px; background-color: #FFD600;">
</div>

In [27]:
# =========================================================================
# Step 8:
# BUILD MODEL INPUTS FROM A VIDEO SOURCE + 
# RUN INFERENCE + APPLY DECISION THRESHOLD
# (together in a single function)
# =========================================================================

from Functions.challenge_response import *

# --- global decision config ---
SPOOF_IS_POSITIVE = True                   # label==1 was SPOOF
FAKE_THRESHOLD    = VALIDATION_THRESHOLD    # threshold chosen on val, applied to test


#------------------------------------
# Check for a live video
#------------------------------------

# Print detailed results, both verbose as True
print("Check 1:")
check_video("./DATA/SiW-Mv2/Live/Live_360.avi", 
            threshold=FAKE_THRESHOLD, 
            spoof_is_positive=SPOOF_IS_POSITIVE, 
            model=fas_model,
            device=DEVICE,
            verbose_prep=True, verbose_score=True)
print("\n")

# Quiet prep, just show model score result for this video (the sensible default)
print("Check 2:")
check_video("./DATA/SiW-Mv2/Live/Live_360.avi",
            threshold=FAKE_THRESHOLD, 
            spoof_is_positive=SPOOF_IS_POSITIVE, 
            model=fas_model,
            device=DEVICE)
print("\n")

# Fully silent, just grab the returned dict
print("Check 3:")
result = check_video("./DATA/SiW-Mv2/Live/Live_360.avi", 
                     threshold=FAKE_THRESHOLD, 
                     spoof_is_positive=SPOOF_IS_POSITIVE, 
                     model=fas_model,
                     device=DEVICE,
                     verbose_prep=False, verbose_score=False)
print("\n")

#-----------------------------------------------
# loop over live + spoof, both fully verbose
#-----------------------------------------------
print("Check 4:")
for vp in ["./DATA/SiW-Mv2/Live/Live_360.avi",
           "./DATA/SiW-Mv2/Spoof/Makeup_Impersonation/Makeup_Im_20.mov"]:
    check_video(vp,
                threshold=FAKE_THRESHOLD, 
                spoof_is_positive=SPOOF_IS_POSITIVE, 
                model=fas_model,
                device=DEVICE,
                verbose_prep=True, verbose_score=True)
print("\n")

Check 1:
[Live_360.avi]
  Faces found : 16/16
  Quality     : usable_frac=1.00, n_good=13, n_interp=3, n_unfilled=0
  RGB  (1, 16, 3, 224, 224) | Freq (1, 16, 3, 224, 224) | Flow (1, 16, 3, 224, 224)
FAS INFERENCE RESULT
Video            : Live_360.avi
Raw logit        : -4.9413
Positive-class p : 0.0071  (SPOOF_IS_POSITIVE=True)
Spoof prob       : 0.0071
Live prob        : 0.9929
Threshold        : 0.16464646464646465
VERDICT          : LIVE
Clip quality     : usable_frac=1.00


Check 2:
FAS INFERENCE RESULT
Video            : Live_360.avi
Raw logit        : -4.9413
Positive-class p : 0.0071  (SPOOF_IS_POSITIVE=True)
Spoof prob       : 0.0071
Live prob        : 0.9929
Threshold        : 0.16464646464646465
VERDICT          : LIVE
Clip quality     : usable_frac=1.00


Check 3:


Check 4:
[Live_360.avi]
  Faces found : 16/16
  Quality     : usable_frac=1.00, n_good=13, n_interp=3, n_unfilled=0
  RGB  (1, 16, 3, 224, 224) | Freq (1, 16, 3, 224, 224) | Flow (1, 16, 3, 224, 224)
FAS INFERE

<div style="background-color: #FFF9C4; padding: 12px; border-radius: 6px;">
<hr style="border: none; height: 3px; background-color: #FFD600;">
</div>